In [1]:
import numpy as np
import pandas as pd

from PyFairnessAI.model_selection import RandomizedSearchCVFairness
from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

from sklearn.model_selection import train_test_split, StratifiedKFold

from aif360.datasets import GermanDataset

from PyMachineLearning.models import LogisticRegressionThreshold

pd.set_option('display.max_colwidth', None)

pip install 'aif360[inFairness]'
pip install 'aif360[OptimalTransport]'
pip install 'aif360[FairAdapt]'
pip install 'aif360[LFR]'


In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [4]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=111)
log_reg_estimator = LogisticRegressionThreshold(solver='liblinear', random_state=123, max_iter=200)

In [6]:
param_grid = {
    'penalty': ['l1', 'l2'], 
    'C': [0.01, 0.1, 1, 10, 30, 50, 75, 100],  # Regularization strength
    'class_weight': ['balanced', None],
    'threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

In [9]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='fairness', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [53]:
fairness_random_search.cv_results_

,params,predictive-score,fairness-score,combined-score
24,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
17,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
33,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
27,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.8}",0.503361,0.005462,0.559204
15,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.3}",0.556303,0.044634,0.590921
19,"{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
34,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
38,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
16,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.5}",0.552941,0.063679,0.551432


In [54]:
fairness_random_search.best_params_

{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}

In [55]:
fairness_random_search.best_score_

0.0

In [57]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='predictive', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [58]:
fairness_random_search.cv_results_

,params,predictive-score,fairness-score,combined-score
7,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}",0.731933,0.265248,0.538344
42,"{'penalty': 'l1', 'C': 100, 'class_weight': None, 'threshold': 0.8}",0.731373,0.268584,0.531481
46,"{'penalty': 'l2', 'C': 100, 'class_weight': None, 'threshold': 0.8}",0.729412,0.265029,0.533968
30,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.7}",0.721569,0.258431,0.530653
43,"{'penalty': 'l1', 'C': 30, 'class_weight': None, 'threshold': 0.7}",0.721569,0.258431,0.530653
35,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.7}",0.720728,0.256170,0.533002
36,"{'penalty': 'l1', 'C': 50, 'class_weight': 'balanced', 'threshold': 0.7}",0.720728,0.256170,0.533002
13,"{'penalty': 'l2', 'C': 1, 'class_weight': None, 'threshold': 0.7}",0.717087,0.278647,0.487010
11,"{'penalty': 'l2', 'C': 100, 'class_weight': None, 'threshold': 0.7}",0.714566,0.261651,0.511835
37,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced', 'threshold': 0.5}",0.713445,0.283352,0.471951


In [59]:
fairness_random_search.best_params_

{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced', 'threshold': 0.6}

In [60]:
fairness_random_search.best_score_

0.7319327731092438

In [12]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='combined', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=50, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [13]:
fairness_random_search.cv_results_

,params,predictive-score,fairness-score,combined-score
34,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
19,"{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
38,"{'penalty': 'l2', 'C': 30, 'class_weight': None, 'threshold': 0.2}",0.572549,0.055936,0.601905
15,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.3}",0.556303,0.044634,0.590921
17,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
33,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.9}",0.500000,0.000000,0.562368
24,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None, 'threshold': 0.3}",0.500000,0.000000,0.562368
27,"{'penalty': 'l2', 'C': 0.01, 'class_weight': 'balanced', 'threshold': 0.8}",0.503361,0.005462,0.559204
16,"{'penalty': 'l2', 'C': 0.01, 'class_weight': None, 'threshold': 0.5}",0.552941,0.063679,0.551432


In [14]:
fairness_random_search.best_params_

{'penalty': 'l2', 'C': 75, 'class_weight': None, 'threshold': 0.2}

In [15]:
fairness_random_search.best_score_

0.6019046698635167